# **Product Similarity** 🔎

Este notebook tem como objetivo construir uma abordagem de **recomendação produto → produto** baseada em similaridade entre itens.

A pergunta central desta etapa é:

> Dado um produto selecionado, quais outros produtos apresentam padrões de compra mais semelhantes?

Diferente da análise de cesta de compras, que observa produtos frequentemente comprados juntos no mesmo pedido, esta abordagem representa cada produto pelo conjunto de pedidos em que aparece e calcula a proximidade entre esses padrões.

Para isso, serão utilizados principalmente:

- uma **matriz pedido-produto** em formato esparso;
- a métrica de **similaridade cosseno** entre produtos;
- uma função de recomendação baseada em produtos semelhantes.

💡 **Observação**:

Esta etapa complementa a análise de Market Basket. Enquanto a análise de associação busca relações diretas de coocorrência, a similaridade entre produtos ajuda a encontrar itens com comportamento de compra parecido dentro da base.


In [1]:
# =========================================
# 📚 IMPORTAÇÃO DAS BIBLIOTECAS
# =========================================

import gc
import warnings

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
# =========================================
# 📂 CARREGAMENTO DOS DADOS
# =========================================

DATA_PATH = Path("../data/raw")

order_products_prior = pd.read_csv(
    DATA_PATH / "order_products__prior.csv"
)

products = pd.read_csv(
    DATA_PATH / "products.csv"
)

aisles = pd.read_csv(
    DATA_PATH / "aisles.csv"
)

departments = pd.read_csv(
    DATA_PATH / "departments.csv"
)

print("✅ Dados carregados com sucesso!")
print("order_products_prior:", order_products_prior.shape)
print("products:", products.shape)

✅ Dados carregados com sucesso!
order_products_prior: (32434489, 4)
products: (49688, 4)


## 📋 **1. Preparação do Ambiente e Carregamento dos Dados**

Nesta etapa, serão importadas as bibliotecas necessárias e carregadas as bases utilizadas para calcular a similaridade entre produtos.

As tabelas principais utilizadas são:

- `order_products_prior`: histórico de produtos comprados nos pedidos anteriores;
- `products`: cadastro dos produtos;
- `aisles`: corredores/categorias intermediárias;
- `departments`: departamentos dos produtos.

💡 **Observação**:

O conjunto `prior` será utilizado como base de comportamento histórico, permitindo identificar padrões de compra recorrentes entre os produtos.


## 🛒 **2. Enriquecimento dos Produtos e Amostragem**

Nesta etapa, os produtos serão enriquecidos com informações de corredor e departamento para facilitar a interpretação dos resultados.

Também será criada uma amostra de pedidos e selecionado um conjunto dos produtos mais frequentes, reduzindo o custo computacional da matriz de similaridade.

💡 **Observação**:

A similaridade produto → produto pode ser custosa quando calculada para todos os itens da base. Por isso, limitar a análise aos produtos mais frequentes mantém o notebook mais leve e reduz ruído de itens muito raros.


In [3]:
# =========================================
# 🛒 PRODUTOS ENRIQUECIDOS
# =========================================

products_enriched = (
    products
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

products_enriched.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [4]:
# =========================================
# ⚙️ AMOSTRAGEM DOS PEDIDOS
# =========================================

N_SAMPLE_ORDERS = 50_000
RANDOM_STATE = 42

sample_order_ids = (
    order_products_prior["order_id"]
    .drop_duplicates()
    .sample(
        n=N_SAMPLE_ORDERS,
        random_state=RANDOM_STATE
    )
)

sample_order_products = order_products_prior[
    order_products_prior["order_id"].isin(sample_order_ids)
].copy()

print("Shape da amostra:", sample_order_products.shape)

sample_order_products.head()

Shape da amostra: (505250, 4)


,order_id,product_id,add_to_cart_order,reordered
125,14,20392,1,1
126,14,27845,2,1
127,14,162,3,1
128,14,2452,4,1
129,14,8575,5,1


In [5]:
# =========================================
# 🏆 PRODUTOS MAIS FREQUENTES
# =========================================

TOP_N_PRODUCTS = 1_000

top_product_ids = (
    sample_order_products["product_id"]
    .value_counts()
    .head(TOP_N_PRODUCTS)
    .index
)

sample_order_products = sample_order_products[
    sample_order_products["product_id"].isin(top_product_ids)
].copy()

print("Shape após filtro de produtos:", sample_order_products.shape)
print("Quantidade de produtos únicos:", sample_order_products["product_id"].nunique())
print("Quantidade de pedidos únicos:", sample_order_products["order_id"].nunique())

Shape após filtro de produtos: (272897, 4)
Quantidade de produtos únicos: 1000
Quantidade de pedidos únicos: 46057


## 🧩 **3. Construção da Matriz Pedido-Produto**

Nesta etapa, cada pedido será representado como uma linha e cada produto como uma coluna.

Quando um produto aparece em determinado pedido, a matriz recebe valor `1`; caso contrário, o valor permanece vazio na estrutura esparsa.

💡 **Observação**:

Como a maior parte dos pedidos contém apenas uma pequena fração dos produtos disponíveis, a matriz é naturalmente esparsa. O uso de `csr_matrix` ajuda a reduzir o consumo de memória e melhora a eficiência do processamento.


In [6]:
# =========================================
# 🧩 ÍNDICES DE PEDIDOS E PRODUTOS
# =========================================

order_ids = sample_order_products["order_id"].unique()
product_ids = sample_order_products["product_id"].unique()

order_id_to_idx = {
    order_id: idx
    for idx, order_id in enumerate(order_ids)
}

product_id_to_idx = {
    product_id: idx
    for idx, product_id in enumerate(product_ids)
}

idx_to_product_id = {
    idx: product_id
    for product_id, idx in product_id_to_idx.items()
}

sample_order_products["order_idx"] = (
    sample_order_products["order_id"]
    .map(order_id_to_idx)
)

sample_order_products["product_idx"] = (
    sample_order_products["product_id"]
    .map(product_id_to_idx)
)

sample_order_products.head()

,order_id,product_id,add_to_cart_order,reordered,order_idx,product_idx
126,14,27845,2,1,0,0
128,14,2452,4,1,0,1
131,14,39475,7,1,0,2
134,14,20995,10,1,0,3
135,14,45066,11,0,0,4


In [7]:
# =========================================
# 🧱 MATRIZ PEDIDO-PRODUTO
# =========================================

order_product_matrix = csr_matrix(
    (
        np.ones(len(sample_order_products)),
        (
            sample_order_products["order_idx"],
            sample_order_products["product_idx"]
        )
    ),
    shape=(
        len(order_ids),
        len(product_ids)
    )
)

print("Shape da matriz:", order_product_matrix.shape)
print("Quantidade de valores não-zero:", order_product_matrix.nnz)

Shape da matriz: (46057, 1000)
Quantidade de valores não-zero: 272897


## 🔎 **4. Cálculo da Similaridade Cosseno**

Nesta etapa, será calculada a similaridade cosseno entre os produtos a partir da matriz pedido-produto transposta.

Com isso, produtos que aparecem em padrões de pedidos parecidos tendem a receber scores de similaridade mais altos.

💡 **Observação**:

A diagonal da matriz é zerada para evitar que cada produto seja recomendado como similar a ele mesmo.


In [8]:
# =========================================
# 🔎 COSINE SIMILARITY ENTRE PRODUTOS
# =========================================

product_similarity_matrix = cosine_similarity(
    order_product_matrix.T
)

np.fill_diagonal(product_similarity_matrix, 0)

print("Shape da matriz de similaridade:", product_similarity_matrix.shape)

Shape da matriz de similaridade: (1000, 1000)


## 📋 **5. Seleção e Interpretação das Similaridades**

Nesta etapa, serão selecionados os produtos mais similares para cada item analisado.

Em seguida, os IDs serão enriquecidos com os nomes dos produtos, tornando os resultados mais fáceis de interpretar.

💡 **Observação**:

O filtro de similaridade mínima ajuda a remover associações muito fracas, deixando a tabela final mais útil para recomendação.


In [9]:
# =========================================
# 📋 TOP SIMILARIDADES POR PRODUTO
# =========================================

TOP_K_SIMILAR = 20
MIN_SIMILARITY = 0.02

similarity_rows = []

for product_idx in range(product_similarity_matrix.shape[0]):
    similarities = product_similarity_matrix[product_idx]

    top_indices = np.argsort(similarities)[::-1][:TOP_K_SIMILAR]

    product_a_id = idx_to_product_id[product_idx]

    for similar_idx in top_indices:
        similarity_score = similarities[similar_idx]

        if similarity_score >= MIN_SIMILARITY:
            product_b_id = idx_to_product_id[similar_idx]

            similarity_rows.append(
                {
                    "product_a_id": product_a_id,
                    "product_b_id": product_b_id,
                    "cosine_similarity": similarity_score
                }
            )

product_similarity_rules = pd.DataFrame(similarity_rows)

print("Shape das regras de similaridade:", product_similarity_rules.shape)

product_similarity_rules.head()

Shape das regras de similaridade: (19777, 3)


,product_a_id,product_b_id,cosine_similarity
0,27845,21137,0.133514
1,27845,24852,0.125721
2,27845,13176,0.112578
3,27845,22035,0.099313
4,27845,6104,0.089571


In [10]:
# =========================================
# 🏷️ NOMES DOS PRODUTOS
# =========================================

product_name_map = dict(
    zip(
        products_enriched["product_id"],
        products_enriched["product_name"]
    )
)

product_similarity_rules["product_a"] = (
    product_similarity_rules["product_a_id"]
    .map(product_name_map)
)

product_similarity_rules["product_b"] = (
    product_similarity_rules["product_b_id"]
    .map(product_name_map)
)

product_similarity_rules = product_similarity_rules[
    [
        "product_a_id",
        "product_a",
        "product_b_id",
        "product_b",
        "cosine_similarity"
    ]
]

product_similarity_rules.head()

,product_a_id,product_a,product_b_id,product_b,cosine_similarity
0,27845,Organic Whole Milk,21137,Organic Strawberries,0.133514
1,27845,Organic Whole Milk,24852,Banana,0.125721
2,27845,Organic Whole Milk,13176,Bag of Organic Bananas,0.112578
3,27845,Organic Whole Milk,22035,Organic Whole String Cheese,0.099313
4,27845,Organic Whole Milk,6104,Whole Milk Plain Yogurt,0.089571


## 🛠️ **6. Função de Recomendação por Similaridade**

Nesta etapa, será criada uma função para consultar produtos similares a partir do nome de um item.

A função permite controlar parâmetros como:

- quantidade de recomendações retornadas;
- similaridade mínima;
- busca exata ou parcial pelo nome do produto.

💡 **Observação**:

A busca parcial torna a função mais flexível para uso exploratório, principalmente quando o nome completo do produto não é conhecido.


In [11]:
# =========================================
# 🛒 FUNÇÃO DE RECOMENDAÇÃO POR SIMILARIDADE
# =========================================

def recommend_similar_products(
    product_name: str,
    top_n: int = 10,
    min_similarity: float = 0.02
) -> pd.DataFrame:
    """
    Recommend products based on item-item cosine similarity.

    Parameters
    ----------
    product_name : str
        Name or partial name of the product used as input.
    top_n : int
        Number of recommendations to return.
    min_similarity : float
        Minimum cosine similarity required.

    Returns
    -------
    pd.DataFrame
        DataFrame containing recommended products and similarity score.
    """

    product_name_clean = product_name.lower().strip()

    exact_match = products_enriched[
        products_enriched["product_name"]
        .str.lower()
        .str.strip()
        .eq(product_name_clean)
    ]

    if not exact_match.empty:
        matched_product = exact_match.iloc[0]
    else:
        partial_match = products_enriched[
            products_enriched["product_name"]
            .str.lower()
            .str.contains(product_name_clean, regex=False)
        ]

        if partial_match.empty:
            print("Produto não encontrado.")
            return pd.DataFrame()

        matched_product = partial_match.iloc[0]

    product_id = matched_product["product_id"]
    selected_product = matched_product["product_name"]

    recommendations = product_similarity_rules[
        (product_similarity_rules["product_a_id"] == product_id)
        & (product_similarity_rules["cosine_similarity"] >= min_similarity)
    ].copy()

    recommendations = (
        recommendations
        .sort_values(
            by="cosine_similarity",
            ascending=False
        )
        .head(top_n)
    )

    print(f"Produto selecionado: {selected_product}")

    return recommendations[
        [
            "product_b",
            "cosine_similarity"
        ]
    ]

## 🧪 **7. Testes da Função de Recomendação**

Nesta etapa, serão testadas recomendações para produtos populares da base.

O objetivo é verificar se produtos com padrões semelhantes de compra são recomendados corretamente e se os resultados fazem sentido em termos de categoria e comportamento de consumo.


In [12]:
recommend_similar_products(
    product_name="Organic Strawberries",
    top_n=10,
    min_similarity=0.02
)

Produto selecionado: Organic Strawberries


,product_b,cosine_similarity
665,Bag of Organic Bananas,0.185828
666,Organic Hass Avocado,0.177453
667,Organic Raspberries,0.175280
668,Banana,0.153847
669,Organic Baby Spinach,0.146673
670,Organic Blueberries,0.138057
671,Organic Whole Milk,0.133514
672,Organic Cucumber,0.121431
673,Organic Whole String Cheese,0.116730
674,Organic Grape Tomatoes,0.112561


In [13]:
recommend_similar_products(
    product_name="Organic Whole Milk",
    top_n=10,
    min_similarity=0.02
)

Produto selecionado: Organic Whole Milk


,product_b,cosine_similarity
0,Organic Strawberries,0.133514
1,Banana,0.125721
2,Bag of Organic Bananas,0.112578
3,Organic Whole String Cheese,0.099313
4,Whole Milk Plain Yogurt,0.089571
5,Organic Hass Avocado,0.088455
6,Organic Baby Spinach,0.087302
7,Organic Avocado,0.083655
8,Organic Raspberries,0.080019
9,Organic Garlic,0.066727


In [14]:
recommend_similar_products(
    product_name="Banana",
    top_n=10,
    min_similarity=0.02
)

Produto selecionado: Banana


,product_b,cosine_similarity
349,Organic Avocado,0.177510
350,Organic Fuji Apple,0.161182
351,Strawberries,0.160625
352,Organic Strawberries,0.153847
353,Organic Baby Spinach,0.153167
354,Large Lemon,0.148354
355,Cucumber Kirby,0.145008
356,Honeycrisp Apple,0.143900
357,Seedless Red Grapes,0.129889
358,Limes,0.126517


## 💡 **8. Observação sobre os Resultados**

A matriz pedido-produto construída possui muitos valores vazios, pois cada pedido contém apenas uma parte pequena do catálogo analisado.

Por esse motivo, os valores absolutos de similaridade cosseno tendem a ser relativamente baixos. Mesmo assim, os resultados podem ser úteis quando apresentam coerência semântica e comportamental, recomendando produtos de categorias semelhantes ou frequentemente associados em padrões de compra.

💡 **Observação**:

Esta abordagem funciona bem como complemento às regras de associação, especialmente quando o objetivo é capturar produtos parecidos e não apenas produtos comprados juntos na mesma cesta.


## 💾 **9. Exportação das Regras de Similaridade**

Nesta etapa, a tabela final de similaridade será salva em formato `parquet` para uso nas próximas etapas do projeto.

Esse arquivo poderá alimentar modelos, funções de recomendação ou comparações com outras estratégias desenvolvidas no projeto.


In [15]:
# =========================================
# 💾 EXPORTAÇÃO DAS REGRAS DE SIMILARIDADE
# =========================================

OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

product_similarity_rules.to_parquet(
    OUTPUT_PATH / "product_similarity_rules.parquet",
    index=False
)

print("✅ Regras de similaridade salvas com sucesso!")

✅ Regras de similaridade salvas com sucesso!


## 📌 **Conclusão**

Neste notebook foi construída uma abordagem de recomendação produto → produto baseada em **similaridade entre itens**.

A partir de uma amostra de pedidos, foi criada uma matriz esparsa pedido-produto, em que cada produto foi representado pelo conjunto de pedidos em que apareceu. Em seguida, foi calculada a similaridade cosseno entre os produtos.

Os resultados obtidos indicaram recomendações coerentes para produtos populares, especialmente em categorias como frutas, vegetais, orgânicos e laticínios.

Essa abordagem complementa a análise de Market Basket, pois permite recomendar itens com comportamento de compra semelhante, mesmo quando a relação entre eles não aparece apenas como uma regra direta de coocorrência.

📌 **Próximo passo sugerido**:

Comparar as recomendações geradas por similaridade com as recomendações baseadas em associação, avaliando quais estratégias entregam resultados mais úteis para o sistema final de recomendação.
